In [5]:
import fitz
import pandas as pd
import sys
from pathlib import Path
from ttrpg_ocr.classify import classify_pages
from ttrpg_ocr.classify import classify_pages
from ttrpg_ocr.schemas import BookProfile
import yaml


In [15]:

PDF = Path("/Users/arturo/dev/ttrpg-ocr/data/raw/WFRP - Marienburg Sold Down The River.pdf")
OUT = Path("/Users/arturo/dev/ttrpg-ocr/data/raw/samples"); OUT.mkdir(exist_ok=True, parents=True)

doc = fitz.open(PDF)
print(f"Pages: {len(doc)}")


for i in [7, 14,38,69, 80, 100,150]:
    if i < len(doc):
        page = doc[i]
        print(f"\nPage {i}:")
        print(f"Page {i} embedded text: {len(doc[i].get_text().strip())} chars")
        # Physical page size (in points; 72 points = 1 inch)
        print(f"Page size: {page.rect.width} × {page.rect.height} pts")
        print(f"          = {page.rect.width/72:.2f} × {page.rect.height/72:.2f} inches")

        # Embedded images on the page and their actual pixel dimensions
        for img in page.get_images(full=True):
            xref = img[0]
            info = doc.extract_image(xref)
            print(f"  Image: {info['width']} × {info['height']} px, format {info['ext']}")

        pix = doc[i].get_pixmap(dpi=300)
        pix.save(OUT / f"p{i:03d}.png")

        #Explore the page's text blocks
        blocks = page.get_text("blocks")
        df = pd.DataFrame(blocks, columns=["x0", "y0", "x1", "y1", "text", "block_no", "block_type"])
        df["x_center"] = (df["x0"] + df["x1"]) / 2
        df["is_text"] = df["block_type"] == 0
        print(f"page width: {page.rect.width}, height: {page.rect.height}")
        print(df[["x0", "x1", "x_center", "y0", "y1", "is_text", "text"]].head(20))

Pages: 168

Page 7:
Page 7 embedded text: 5523 chars
Page size: 595.0 × 788.0 pts
          = 8.26 × 10.94 inches
  Image: 1122 × 113 px, format jpeg
  Image: 510 × 423 px, format jpeg
page width: 595.0, height: 788.0
           x0          x1    x_center          y0          y1  is_text  \
0   43.847095  283.000488  163.423792   76.724136  735.818542     True   
1  295.847107  539.353882  417.600494   76.918777  524.619080     True   
2  532.487061  540.522156  536.504608  747.651672  762.042664     True   

                                                text  
0  of an attempt to retake the throne. With their...  
1  THE WASTELAND\nOnce the Wasteland didn't deser...  
2                                                7\n  

Page 14:
Page 14 embedded text: 4573 chars
Page size: 594.0 × 787.0 pts
          = 8.25 × 10.93 inches
  Image: 1098 × 111 px, format jpeg
  Image: 352 × 1383 px, format jpeg
page width: 594.0, height: 787.0
           x0          x1    x_center          y0      

In [11]:
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

pdf_path = repo_root / "data" / "raw" / "WFRP - Marienburg Sold Down The River.pdf"
profile_path = repo_root / "config" / "marienburg.yaml"

profile = BookProfile.model_validate(yaml.safe_load(profile_path.read_text()))

decisions = classify_pages(pdf_path, profile)
print(len(decisions), "pages classified")

df = pd.DataFrame([d.model_dump() for d in decisions])
print(df.head())

168 pages classified
   page_num                  strategy               reason  text_chars  \
0         0         PageStrategy.SKIP        explicit drop           0   
1         1         PageStrategy.SKIP        explicit drop         352   
2         2         PageStrategy.SKIP        explicit drop        2556   
3         3  PageStrategy.NATIVE_TEXT  native text present        6044   
4         4  PageStrategy.NATIVE_TEXT  native text present        1983   

   image_count  
0            1  
1           10  
2           10  
3            2  
4            7  


In [12]:
# show all rows in a printable form
print(df.to_string(index=False))

# summary counts by classification
print(df["strategy"].value_counts())
print(df["reason"].value_counts())

# inspect the key columns only
print(df[["page_num", "strategy", "reason", "text_chars", "image_count"]])

 page_num                 strategy                       reason  text_chars  image_count
        0        PageStrategy.SKIP                explicit drop           0            1
        1        PageStrategy.SKIP                explicit drop         352           10
        2        PageStrategy.SKIP                explicit drop        2556           10
        3 PageStrategy.NATIVE_TEXT          native text present        6044            2
        4 PageStrategy.NATIVE_TEXT          native text present        1983            7
        5 PageStrategy.NATIVE_TEXT          native text present        2786            6
        6 PageStrategy.NATIVE_TEXT          native text present        1711            3
        7 PageStrategy.NATIVE_TEXT          native text present        5523            2
        8 PageStrategy.NATIVE_TEXT          native text present        5546            2
        9 PageStrategy.NATIVE_TEXT          native text present        5274            2
       10 PageStrateg

In [13]:
import pandas as pd
df = pd.DataFrame([d.model_dump() for d in decisions])
print(df["strategy"].value_counts())
print(df.groupby("strategy")["text_chars"].describe())

strategy
PageStrategy.NATIVE_TEXT    152
PageStrategy.SKIP            16
Name: count, dtype: int64
                          count         mean          std    min      25%  \
strategy                                                                    
PageStrategy.NATIVE_TEXT  152.0  4623.302632  1423.791394  882.0  3591.75   
PageStrategy.SKIP          16.0   571.500000  1349.906318    0.0     0.00   

                             50%     75%     max  
strategy                                          
PageStrategy.NATIVE_TEXT  4687.0  5597.0  7548.0  
PageStrategy.SKIP            1.0   167.5  4952.0  


In [17]:
from src.ttrpg_ocr.extract_text import _detect_column_count

# Test case 1: Two columns (gaps > min_gap)
x_centers = [100, 150, 300, 350]  # centers at ~125 and ~325, gap ~200
page_width = 500
max_cols = 3
gap_min_pct = 0.1  # min_gap = 50
result = _detect_column_count(x_centers, page_width, max_cols, gap_min_pct)
print(f"Test 1 result: {result}")  # Expected: 2 (one significant gap)

# Test case 2: Single column (no significant gaps)
x_centers = [100, 120, 140, 160]  # small gaps
result = _detect_column_count(x_centers, page_width, max_cols, gap_min_pct)
print(f"Test 2 result: {result}")  # Expected: 1

# Test case 3: Too few centers
x_centers = [100]  # < 2 centers
result = _detect_column_count(x_centers, page_width, max_cols, gap_min_pct)
print(f"Test 3 result: {result}")  # Expected: 1

Test 1 result: 2
Test 2 result: 1
Test 3 result: 1
